In [34]:
import re
from datetime import date

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [35]:
url = "https://www.ccny.cuny.edu/registrar/fall"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
print(f"HTTP status: {response.status_code}")

HTTP status: 403


In [36]:
soup = BeautifulSoup(response.text, "html.parser")
print(soup.title.get_text(strip=True) if soup.title else "No page title")

Just a moment...


In [37]:
tables = soup.find_all("table")
rows = [row for table in tables for row in table.find_all("tr")]
print(f"Tables found: {len(tables)}")
print(f"Rows found: {len(rows)}")

Tables found: 0
Rows found: 0


In [ ]:
month_names = (
    "January|February|March|April|May|June|July|August|"
    "September|October|November|December"
)
date_pattern = re.compile(
    rf"(?:{month_names})\s+\d{{1,2}},\s+2021|\d{{1,2}}[/-]\d{{1,2}}[/-]2021",
    re.IGNORECASE,
)
weekday_pattern = re.compile(
    r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)$",
    re.IGNORECASE,
)

records = []
for row in rows:
    cells = [cell.get_text(" ", strip=True) for cell in row.find_all(["th", "td"])]
    if not cells:
        continue

    date_position = next(
        (position for position, cell in enumerate(cells) if date_pattern.search(cell)),
        None,
    )
    if date_position is None:
        continue

    date_match = date_pattern.search(cells[date_position])
    parsed_date = pd.to_datetime(date_match.group(), errors="coerce")
    if pd.isna(parsed_date):
        continue

    dow_position = next(
        (position for position, cell in enumerate(cells) if weekday_pattern.fullmatch(cell)),
        None,
    )
    dow = cells[dow_position].title() if dow_position is not None else parsed_date.day_name()
    excluded_positions = {date_position}
    if dow_position is not None:
        excluded_positions.add(dow_position)

    text = " ".join(
        cell for position, cell in enumerate(cells) if position not in excluded_positions
    ).strip()
    records.append({"date": parsed_date.date(), "dow": dow, "text": text})

calendar = pd.DataFrame.from_records(records)
calendar = calendar.drop_duplicates("date").set_index("date").sort_index()
calendar.index.name = "date"
calendar.head()

IndexError: list index out of range